# Fraud 3 — CRISP-DM Phases 1–6: Business-to-Deployment Workshop

**Business context:** Travel booking payments

**Problem:** Identify suspicious travel payments before ticket fulfilment.

**Proposed success criterion:** Compare with an all-legitimate baseline; justify a threshold under a 6% review-capacity constraint; discuss customer friction.

**Dataset:** `fraud_3.csv`. These are synthetic teaching records, not real company data.

**Submission deadline: Monday 5 October 2026 at 13:00 (1:00 PM).** Submit your completed team notebook on Brightspace. Phases 1–2 must be reviewed and updated; your main practical work is Phases 3–6. Any unfinished in-class work must be completed before the deadline.


## Phase 1 — Business Understanding and Phase 2 — Data Understanding (review)
Explain the decision, stakeholder, prediction moment, observation unit, key and target. **Review or revise the supplied success criterion if it does not adequately represent the business decision.** State two evidenced findings and two limitations.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
DATA = Path('fraud_3.csv')
if not DATA.exists(): DATA = Path('fraud_3.csv')
df = pd.read_csv(DATA)
print(df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).head(12))
print('Exact duplicates:',df.duplicated().sum())

(2601, 12)


,transaction_id,amount_eur,customer_avg_amount_30d,account_age_days,previous_transactions_24h,channel,country,hour,fraud_flag,post_investigation_loss_eur,days_to_departure,refund_after_trip_eur
0,T6000000,NaN,43.58,1357,4,ONLINE,FR,16,0,0.0,73,0.0
1,T6000001,17.82,43.80,1983,15,POS,FR,9,0,0.0,46,0.0
2,T6000002,143.45,31.24,612,12,Online,DE,9,0,0.0,5,0.0
3,T6000003,47.77,17.09,194,15,online,FR,9,0,0.0,167,0.0
4,T6000004,20.90,14.42,1622,5,Mobile,NL,13,0,0.0,199,0.0


amount_eur                     11
transaction_id                  0
customer_avg_amount_30d         0
account_age_days                0
previous_transactions_24h       0
channel                         0
country                         0
hour                            0
fraud_flag                      0
post_investigation_loss_eur     0
days_to_departure               0
refund_after_trip_eur           0
dtype: int64

Exact duplicates: 1


**Phase 1**

**Stakeholders:** The Head of Payment Risk decides how payments are checked, and the fraud analysts do the checks. Also affected: the customer, customer service and the finance team.

**Decision:** Let a payment go through, or hold it for a manual check before the ticket is issued.

**Prediction moment:** When the payment is authorised, before the ticket is issued.

In [2]:
# Evidence for Phase 2: how do fraud and legitimate payments differ? (medians, because amounts are skewed)
clean = df.drop_duplicates()
print(f"Fraud: {clean['fraud_flag'].sum()} of {len(clean)} payments = {clean['fraud_flag'].mean():.2%}")
display(clean.groupby('fraud_flag')[['amount_eur', 'account_age_days', 'previous_transactions_24h',
                                     'days_to_departure']].median())
print("Rows with post_investigation_loss_eur above 0, per fraud_flag:")
print((clean['post_investigation_loss_eur'] > 0).groupby(clean['fraud_flag']).sum())

Fraud: 58 of 2600 payments = 2.23%


,amount_eur,account_age_days,previous_transactions_24h,days_to_departure
fraud_flag,,,,
0,47.35,1078.5,12.0,106.0
1,68.85,866.0,12.5,103.5


Rows with post_investigation_loss_eur above 0, per fraud_flag:
fraud_flag
0     0
1    58
Name: post_investigation_loss_eur, dtype: int64


### Your answer
**One row represents:** one transaction  
**Dimensions:** 2601 rows / 12 variables (2600 unique transactions, because one row is an exact duplicate)  
**Identifier/business key:** transaction_id  
**Main outcome:** fraud_flag  
**Most relevant variables and why:** `amount_eur` and `account_age_days`. Fraud payments are larger (median €68.85 against €47.35) and come from newer accounts (median 866 days against 1,079). We first thought country and the number of transactions in the last 24 hours would matter most, but the data does not show it: `previous_transactions_24h` is almost the same for fraud and legitimate payments (12.5 against 12), and without a customer ID we cannot see the same person paying from two countries.

**Revision of the success criterion**

Approving every payment would be about 97.8% accurate and catch no fraud at all, so accuracy is not a useful measure here. Fraud is 2.23% of payments and the team can review 6%, so there is enough capacity. What matters is whether the model puts the right payments into that 6%.

Our revised criteria:
1. When the 6% riskiest payments are reviewed, the model catches more fraud than random review and more than a simple amount rule.
2. The hit rate (the share of reviewed payments that are really fraud) is above the 2.2% base rate.
3. The model only uses information that is known before the ticket is issued.

Customer friction is the number of honest customers who are held. We count it, but we cannot put a price on it because the data has no cost information. We do not set a target percentage yet; that should be agreed with the business after a baseline period.

**2 findings and 2 limitations**

2 findings:
1. Fraud is rare: 58 of 2,600 transactions (2.23%). This is why accuracy is misleading.
2. `post_investigation_loss_eur` cannot be used in the model. It is above 0 for all 58 fraud cases and 0 for all other payments, and it is only known AFTER the investigation. `refund_after_trip_eur` is also only known after the trip.

2 limitations:
1. There are only 58 fraud cases, which is very little to learn from, and the data has no date and no customer ID.
2. There is no information on what a false positive (holding an honest customer) or a manual review costs the company.

### Data-quality findings
| Observation/issue | Variable(s) | Evidence | Why it matters |
|---|---|---|---|
| Missing | amount_eur | 11 rows | The amount is an important input for the model, so the gaps have to be handled. |
| Duplicate | transaction_id | 1 exact duplicate row (T6000055 appears twice) | One payment would be counted twice. All values are identical, so it looks like an error. |
| Capital letters | channel | online / Online / ONLINE | The model would treat one channel as three different ones. |

## Phase 3 — Data Preparation: three justified decisions
For EACH insight: evidence → technique → before/after verification → interpretation → next decision. Consider identifiers, leakage, duplicate keys, suspicious values, categories, date parsing and feature engineering. Explain why you retain, delete or combine variables. **Do not use the target or post-outcome fields as predictors.**

In [3]:
work = df.copy()

In [4]:
# [1] Remove the exact duplicate row
before_rows = work.shape[0]
work = work.drop_duplicates()
after_rows = work.shape[0]
print(f"[1] Duplicates removed: {before_rows - after_rows}  (before={before_rows}, after={after_rows})")

# [2] Standardise inconsistent 'channel' categories (same channel, different capitalisation)
print("\n[2] channel BEFORE:")
print(work['channel'].value_counts())
work['channel'] = work['channel'].str.strip().str.capitalize()
print("\n[2] channel AFTER:")
print(work['channel'].value_counts())

# [3] Handle missing amount_eur values: keep a missing-flag, then impute with the median
missing_before = work['amount_eur'].isna().sum()
work['amount_eur_missing'] = work['amount_eur'].isna().astype(int)
median_amount = work['amount_eur'].median()
work['amount_eur'] = work['amount_eur'].fillna(median_amount)
print(f"\n[3] amount_eur missing BEFORE: {missing_before}  AFTER: {work['amount_eur'].isna().sum()}  (filled with median={median_amount:.2f})")

# [4] New feature (feature engineering): how far the amount is from the customer's usual spend
work['amount_to_avg_ratio'] = work['amount_eur'] / work['customer_avg_amount_30d'].replace(0, np.nan)
work['amount_to_avg_ratio'] = work['amount_to_avg_ratio'].fillna(work['amount_to_avg_ratio'].median())
print("\n[4] Mean ratio by fraud_flag:")
print(work.groupby('fraud_flag')['amount_to_avg_ratio'].mean().round(2))

# [5] Exclude identifier and leakage columns from the model (not deleted, just not used as predictors)
excluded_cols = ['transaction_id', 'post_investigation_loss_eur', 'refund_after_trip_eur']
print("\n[5] Columns excluded from the model (identifier / leakage):", excluded_cols)

work.head()

[1] Duplicates removed: 1  (before=2601, after=2600)

[2] channel BEFORE:
channel
Mobile    661
POS       654
online    644
Online    634
ONLINE      7
Name: count, dtype: int64

[2] channel AFTER:
channel
Online    1285
Mobile     661
Pos        654
Name: count, dtype: int64

[3] amount_eur missing BEFORE: 11  AFTER: 0  (filled with median=47.52)

[4] Mean ratio by fraud_flag:
fraud_flag
0    2.41
1    3.93
Name: amount_to_avg_ratio, dtype: float64

[5] Columns excluded from the model (identifier / leakage): ['transaction_id', 'post_investigation_loss_eur', 'refund_after_trip_eur']


,transaction_id,amount_eur,customer_avg_amount_30d,account_age_days,previous_transactions_24h,channel,country,hour,fraud_flag,post_investigation_loss_eur,days_to_departure,refund_after_trip_eur,amount_eur_missing,amount_to_avg_ratio
0,T6000000,47.52,43.58,1357,4,Online,FR,16,0,0.0,73,0.0,1,1.090408
1,T6000001,17.82,43.80,1983,15,Pos,FR,9,0,0.0,46,0.0,0,0.406849
2,T6000002,143.45,31.24,612,12,Online,DE,9,0,0.0,5,0.0,0,4.591869
3,T6000003,47.77,17.09,194,15,Online,FR,9,0,0.0,167,0.0,0,2.795202
4,T6000004,20.90,14.42,1622,5,Mobile,NL,13,0,0.0,199,0.0,0,1.449376


### Preparation decision log
| Phase 2 finding | Technique and reason | Verification | Risk | Next decision |
|---|---|---|---|---|
| 1 exact duplicate row (`transaction_id`) | Removed with `drop_duplicates()` | Rows: 2,601 → 2,600 | If it were a real repeated purchase we'd lose data, but identical values make an error more likely | Keep the deduplicated set |
| `channel` had 5 inconsistent labels (case only) | Standardised with `.str.capitalize()` | Categories: 5 → 3 (`Mobile`, `Online`, `Pos`) | Assumes casing was the only difference (confirmed) | One-hot encode the clean 3 categories |
| `amount_eur` had 11 missing values | Added a missing-flag + imputed with the median (€47.52) | Missing: 11 → 0 | Imputation hides the real value and why it was missing | In the end we did not use the flag in the model (only 11 rows, none of them fraud) |
| `post_investigation_loss_eur` and `refund_after_trip_eur` reveal the outcome / are post-decision | Excluded from the model entirely, along with `transaction_id` | Feature list built without these 3 columns | We lose the loss amount as a model input, but we still use it in Phase 5 to see how much fraud loss we catch | Model only with pre-decision information |

## Phase 4 — Modelling
Define the prediction moment and allowed predictors. Use a defensible train/test split (chronological for forecasting). Fit learned preprocessing on TRAINING data only. Compare a simple baseline with a candidate model. Explain your feature engineering and excluded fields.

In [5]:
# Build and test your baseline and candidate model here.
# Suggested tools: sklearn Pipeline, ColumnTransformer, SimpleImputer,
# OneHotEncoder, LogisticRegression, RandomForestRegressor or Ridge.
# For forecasting: use time-ordered validation; no future sales or actual future footfall.
# Phase 4 — Modelling
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

target = 'fraud_flag'
numeric_features = ['amount_eur','customer_avg_amount_30d','account_age_days',
                     'previous_transactions_24h','hour','days_to_departure','amount_to_avg_ratio']
categorical_features = ['channel','country']
# transaction_id, post_investigation_loss_eur, refund_after_trip_eur are excluded (leakage, Phase 3)

X = work[numeric_features + categorical_features]
y = work[target]

# Stratified split: fraud is only ~2.2% of the data, so stratify keeps that ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} rows, fraud rate {y_train.mean():.4f}")
print(f"Test:  {X_test.shape[0]} rows, fraud rate {y_test.mean():.4f}  ({y_test.sum()} fraud cases)")

# Preprocessing is fit ONLY on training data (inside the Pipeline), to avoid leakage
preprocess = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

# --- Baseline: always predict "legitimate" ---
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
print("\n=== Baseline: always predict legitimate ===")
print(confusion_matrix(y_test, baseline_pred))
print(classification_report(y_test, baseline_pred, digits=3, zero_division=0))

# --- Candidate model: Logistic Regression, class-weighted for the imbalance ---
candidate = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
candidate.fit(X_train, y_train)
candidate_pred = candidate.predict(X_test)
candidate_proba = candidate.predict_proba(X_test)[:, 1]

print("\n=== Candidate: Logistic Regression (class-weighted), default 0.5 threshold ===")
print(confusion_matrix(y_test, candidate_pred))
print(classification_report(y_test, candidate_pred, digits=3, zero_division=0))
print("ROC AUC:", round(roc_auc_score(y_test, candidate_proba), 3))

# --- Review-capacity check: review only the top 6% riskiest transactions ---
review_capacity = 0.06
n_review = max(1, int(np.ceil(len(X_test) * review_capacity)))
ranked = pd.DataFrame({'y_true': y_test.values, 'proba': candidate_proba}).sort_values('proba', ascending=False)
reviewed = ranked.head(n_review)
fraud_caught = int(reviewed['y_true'].sum())
total_fraud_in_test = int(y_test.sum())
print(f"\n=== Review-capacity check: reviewing the top {review_capacity:.0%} ({n_review} of {len(X_test)} test transactions) ===")
print(f"Fraud cases caught: {fraud_caught} of {total_fraud_in_test} total fraud in the test set "
      f"({fraud_caught/total_fraud_in_test:.1%} recall at {review_capacity:.0%} capacity)")

Train: 1820 rows, fraud rate 0.0225
Test:  780 rows, fraud rate 0.0218  (17 fraud cases)

=== Baseline: always predict legitimate ===
[[763   0]
 [ 17   0]]
              precision    recall  f1-score   support

           0      0.978     1.000     0.989       763
           1      0.000     0.000     0.000        17

    accuracy                          0.978       780
   macro avg      0.489     0.500     0.494       780
weighted avg      0.957     0.978     0.967       780


=== Candidate: Logistic Regression (class-weighted), default 0.5 threshold ===
[[481 282]
 [  9   8]]
              precision    recall  f1-score   support

           0      0.982     0.630     0.768       763
           1      0.028     0.471     0.052        17

    accuracy                          0.627       780
   macro avg      0.505     0.550     0.410       780
weighted avg      0.961     0.627     0.752       780

ROC AUC: 0.52

=== Review-capacity check: reviewing the top 6% (47 of 780 test transac

### Phase 4 — Modelling: written answer

**Prediction moment and allowed predictors:** The prediction happens at payment authorisation, before the ticket is issued. Allowed predictors are listed in `numeric_features` and `categorical_features` in the code above; `transaction_id`, `post_investigation_loss_eur` and `refund_after_trip_eur` are excluded because they are only known after the outcome.

**Train/test split:** We used a stratified 70/30 split (a split by date would be more realistic, but the data has no date). Stratifying was necessary because fraud is only 2.2% of the data — without it, the test set could randomly end up with very few or no fraud cases. The output confirms both sets kept a similar fraud rate (train 2.25%, test 2.18%).

**Preprocessing fit on training data only:** The `ColumnTransformer`  is placed inside the `Pipeline` and only called via `candidate.fit(X_train, y_train)`, so it learns its parameters exclusively from the training set, never from the test set. One exception we noticed later: in Phase 3 we filled the 11 missing amounts with the median of all rows, before the split. We fix this in round 2.

**Baseline vs. candidate:** The baseline reaches 97.8% accuracy but 0% recall, it catches no fraud at all. The candidate Logistic Regression catches 47% of the fraud at the default 0.5 threshold, but only because it flags 290 of 780 payments (37%), far more than the 6% the team can review. At the 6% capacity it catches 1 of 17, which is what random review would catch. With a ROC AUC of 0.52 this first model is not better than guessing.

**Feature engineering and excluded fields:** We engineered `amount_to_avg_ratio` (amount divided by the customer's 30-day average spend) in Phase 3, which was higher for fraud than for legitimate transactions (mean 3.93 vs 2.41), but this was not enough to make the model useful. `transaction_id`, `post_investigation_loss_eur` and `refund_after_trip_eur` were excluded as they are not available at the prediction moment.

### Round 2: back to Phase 3 and Phase 4
Our first model was no better than random review, so we went back and changed the preparation. This is the CRISP-DM loop: the evaluation sends you back to an earlier phase.

| What we saw in round 1 | What we changed | Why |
|---|---|---|
| 9 input fields (14 after encoding) but only 41 fraud cases in the training data | Only three features: amount compared with the customer's usual amount, new account, night booking | Too many inputs for so few fraud cases, so the model learns noise |
| The median for missing amounts was calculated on all rows | The pipeline fills the gaps using training rows only | Test rows must not influence anything that is learned |
| `capitalize()` turned POS into Pos | `upper()` | Keeps the label correct |
| Country was an input | Country is left out | It showed no clear pattern, and it is close to nationality, which is unfair to use |

In [6]:
# ROUND 2 - back to Phase 3 (prepare again) and Phase 4 (model again)
work2 = df.drop_duplicates().copy()
work2['channel'] = work2['channel'].str.strip().str.upper()      # upper() keeps POS as POS

# Three features we build ourselves. Each is a fixed rule per row, so nothing is learned from the data.
work2['amount_vs_usual'] = np.log(work2['amount_eur'] / work2['customer_avg_amount_30d'])  # how unusual for this customer
work2['new_account'] = (work2['account_age_days'] <= 30).astype(int)                       # account 30 days old or less
work2['night'] = (work2['hour'] <= 5).astype(int)                                          # booked 00:00-05:59
work2['amount_to_avg_ratio'] = work2['amount_eur'] / work2['customer_avg_amount_30d']      # only to re-test model 1 later
features2 = ['amount_vs_usual', 'new_account', 'night']

# Same split as round 1. Missing amounts are NOT filled here: the pipeline fills them using training rows only.
train2, test2 = train_test_split(work2, test_size=0.30, random_state=42, stratify=work2['fraud_flag'])
print(f"Train: {len(train2)} rows, {train2['fraud_flag'].sum()} fraud | Test: {len(test2)} rows, {test2['fraud_flag'].sum()} fraud")
print("Fraud rate in the TRAINING rows:")
print("  new accounts:", f"{train2[train2['new_account'] == 1]['fraud_flag'].mean():.1%}",
      "| older accounts:", f"{train2[train2['new_account'] == 0]['fraud_flag'].mean():.1%}")
print("  night bookings:", f"{train2[train2['night'] == 1]['fraud_flag'].mean():.1%}",
      "| rest of day:", f"{train2[train2['night'] == 0]['fraud_flag'].mean():.1%}")

model2 = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
model2.fit(train2[features2], train2['fraud_flag'])
test2 = test2.assign(score=model2.predict_proba(test2[features2])[:, 1])

def review_top(data, score_column, capacity=0.06):
    """Return the payments the team would review: the highest scores, up to the capacity."""
    n = int(np.ceil(len(data) * capacity))
    return data.sort_values(score_column, ascending=False, kind='stable').head(n)

reviewed2 = review_top(test2, 'score')
total_fraud = test2['fraud_flag'].sum()

# A simple rule without a model: review the payments that are largest compared with the customer's usual amount
test2['rule_score'] = test2['amount_vs_usual'].fillna(train2['amount_vs_usual'].median())
reviewed_rule = review_top(test2, 'rule_score')

print(f"\nReviewing the top 6% = {len(reviewed2)} of {len(test2)} test payments ({total_fraud} fraud cases in total)")
print(f"  Approve everything:        0 caught")
print(f"  Random review (expected):  {len(reviewed2) * total_fraud / len(test2):.1f} caught")
print(f"  Simple amount rule:        {reviewed_rule['fraud_flag'].sum()} caught")
print(f"  Model 1 (round 1, above):  {fraud_caught} caught")
print(f"  Model 2 (three features):  {reviewed2['fraud_flag'].sum()} caught")
print("  Model 2 ROC AUC:", round(roc_auc_score(test2['fraud_flag'], test2['score']), 3))

Train: 1820 rows, 41 fraud | Test: 780 rows, 17 fraud
Fraud rate in the TRAINING rows:
  new accounts: 12.0% | older accounts: 2.1%
  night bookings: 3.5% | rest of day: 1.8%

Reviewing the top 6% = 47 of 780 test payments (17 fraud cases in total)
  Approve everything:        0 caught
  Random review (expected):  1.0 caught
  Simple amount rule:        0 caught
  Model 1 (round 1, above):  1 caught
  Model 2 (three features):  4 caught
  Model 2 ROC AUC: 0.593


**Round 2 result.** In the training rows, new accounts have 12.0% fraud against 2.1% for older accounts, and night bookings 3.5% against 1.8%. With these three features the model catches 4 of 17 fraud cases in the top 6%. Our first model caught 1, random review about 1, and a simple amount rule 0. The ROC AUC is 0.593 (it was 0.52), so the model is better but still weak.

## Phase 5 — Evaluation
Report model vs baseline on held-out observations. Interpret relevant classification errors or forecasting MAE by business segment. Compare with your Phase 1 success criterion and discuss whether that criterion is realistic. If a required cost/impact is unavailable, state the assumption or request more data.

In [7]:
# Evidence for Phase 5 (model 2, test set, top 6% reviewed)
caught = reviewed2['fraud_flag'].sum()
honest_held = len(reviewed2) - caught
missed = total_fraud - caught
loss_total = test2['post_investigation_loss_eur'].sum()      # used for evaluation only, never as a model input
loss_caught = reviewed2['post_investigation_loss_eur'].sum()

print(f"Fraud caught: {caught} | fraud missed: {missed} | honest customers held: {honest_held} "
      f"({honest_held / caught:.1f} per fraud caught)")
print(f"Hit rate: {caught / len(reviewed2):.1%} | base rate: {test2['fraud_flag'].mean():.1%}")
print(f"Fraud loss in test set: EUR {loss_total:.2f} | in reviewed payments: EUR {loss_caught:.2f} "
      f"| share missed: {1 - loss_caught / loss_total:.0%}")
print(f"Break-even cost per review: EUR {loss_caught / len(reviewed2):.2f}")

print("\nWho is held?")
for name, column in [('New accounts', 'new_account'), ('Night bookings', 'night')]:
    group = test2[test2[column] == 1]
    held = reviewed2[reviewed2[column] == 1]
    print(f"  {name}: {len(group)} in test set, {len(held)} held, of which {(held['fraud_flag'] == 0).sum()} honest")

print("\nWhat if the team reviewed 10% instead of 6%?")
print(f"  Fraud caught at 10%: {review_top(test2, 'score', 0.10)['fraud_flag'].sum()}")

# Would a fixed cut-off keep the queue at 6%? (needed for Phase 6)
train_scores = model2.predict_proba(train2[features2])[:, 1]
cutoff = np.quantile(train_scores, 0.94)
print(f"\nCut-off that holds 6% of training payments: {(train_scores >= cutoff).mean():.1%} of train, "
      f"{(test2['score'] >= cutoff).mean():.1%} of test")

# Did the new preparation help? Compare both sets of inputs on the training rows (5 folds, repeated 10 times)
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.base import clone
model1 = Pipeline([('prep', preprocess), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))])
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
results = {'Model 1 inputs (9 fields)': [], 'Model 2 inputs (3 features)': []}
for fit_rows, check_rows in cv.split(train2, train2['fraud_flag']):
    fit_part, check_part = train2.iloc[fit_rows], train2.iloc[check_rows]
    for name, model, cols in [('Model 1 inputs (9 fields)', model1, numeric_features + categorical_features),
                              ('Model 2 inputs (3 features)', model2, features2)]:
        m = clone(model).fit(fit_part[cols], fit_part['fraud_flag'])
        scored = check_part.assign(s=m.predict_proba(check_part[cols])[:, 1])
        results[name].append(review_top(scored, 's')['fraud_flag'].sum() / check_part['fraud_flag'].sum())
print("\nAverage share of fraud caught at 6% (cross-validation on training rows):")
for name, values in results.items():
    print(f"  {name}: {np.mean(values):.1%}")

Fraud caught: 4 | fraud missed: 13 | honest customers held: 43 (10.8 per fraud caught)
Hit rate: 8.5% | base rate: 2.2%
Fraud loss in test set: EUR 1048.99 | in reviewed payments: EUR 250.09 | share missed: 76%
Break-even cost per review: EUR 5.32

Who is held?
  New accounts: 13 in test set, 13 held, of which 11 honest
  Night bookings: 205 in test set, 36 held, of which 34 honest

What if the team reviewed 10% instead of 6%?
  Fraud caught at 10%: 4

Cut-off that holds 6% of training payments: 6.0% of train, 9.6% of test



Average share of fraud caught at 6% (cross-validation on training rows):
  Model 1 inputs (9 fields): 14.0%
  Model 2 inputs (3 features): 17.8%


1. The test set has 780 payments, 17 of them fraud. The team can review 6%, so 47 payments. Among those 47, our second model found 4 fraud cases. Approving everything finds 0, picking 47 at random finds about 1, and our first model found 1. So the model is better than guessing, but it still misses 13 of 17.

2. The 13 missed cases carry 76% of the fraud loss. 43 honest customers were held for nothing, about 11 per fraud case caught. New customers are hit hardest: all 13 new accounts were held, and 11 of them were honest. For a travel company, making new customers wait on their first booking is costly.

3. We have met our Phase 1 criteria. The model caught 4, against about 1 for random review and 0 for a simple amount rule. Its hit rate is 8.5% (4 of 47), above the 2.2% base rate. It only uses information known before the ticket is issued. The 6% capacity is realistic and the limit is the data, because reviewing 10% instead of 6% caught no extra fraud.

4. We cannot say if it saves money. The data has no cost per review and no cost for holding an honest customer. As a rough guide, the 47 reviews covered €250.09 of fraud loss, so a review must cost less than about €5.32 to pay off. We would ask the business for both costs.

5. We are not very sure, because the test set has only 17 fraud cases. One case changes the result by 5.9 points. Our new preparation did help (fraud caught rose from 14.0% to 17.8% in cross-validation), but we need more data before real bookings are held, so we go back to Phase 2.

## Phase 6 — Deployment proposal (design only)
Specify who uses the prediction, when it runs, what action it supports, a human override, monitoring metric, feedback/refresh trigger and a condition for pausing use. A diagram or 6–8 sentences is sufficient. No production system required.

The Head of Payment Risk owns the model and the fraud analysts use it. It scores each payment at authorisation, before the ticket is issued.

Payments above the cut-off are held for review, with the queue capped at 6%. The model never declines a payment by itself. An analyst releases, verifies or cancels, can overrule the score, and reviews the nearest departures first.

We monitor the weekly hit rate of the queue (8.5% in our test, 2.2% base rate). The analysts' decisions and the later investigation results are saved, so we can check the model and retrain it. We retrain if the share of payments held leaves 4–8%, since the same cut-off held 6.0% of training payments but 9.6% of test payments.

We pause and return to the current process if the hit rate stays at or below the base rate for four weeks.

We first run the model in the background without holding any bookings, and we leave country out. Each week we check which groups of honest customers are being held most.

## Final group decision + individual accountability
**Group:** Pilot / Improve / Revisit earlier CRISP-DM phase / Do not deploy yet. Cite evidence.

**Individual:** Every member adds a named Markdown paragraph describing one preparation choice, one model finding and one business implication.

### Group decision: do not deploy yet. Go back to Phase 2 for more data and test the model in the background.

**What speaks for the model:** it catches 4 of 17 fraud cases at the 6% capacity, against about 1 for random review. Its hit rate is 8.5% against a 2.2% base rate. It only uses information known before the ticket is issued.

**What speaks against it:** it misses 13 of 17 fraud cases (76% of the fraud loss). It holds 43 honest customers to catch 4 fraud cases, and every new account is held. All of this is based on only 17 fraud cases.

**Why more data and not more modelling:** we already improved the preparation (from 1 to 4 caught), and reviewing more payments does not help (10% catches the same 4). What is missing is information.

**What we ask the business for:** the cost of a manual review, the cost of holding an honest customer, a date and a customer ID per payment, and more confirmed fraud cases.

### Individual accountability

**Anisah Sakura Sutrisno (Phases 5 and 6).** *Preparation choice:* When I evaluated our first model, it caught 1 of 17 fraud cases at the 6% capacity, the same as random review. Because of that we went back to Phase 3 and cut the inputs from nine fields to three features, since 41 fraud cases are too few to learn from many inputs. *Model finding:* The second model caught 4 of 17 with a hit rate of 8.5%, but all 13 new accounts in the test set were held and 11 of them were honest. *Business implication:* The model is better than guessing, but it would make every new customer wait on their first booking. So in Phase 6 I proposed to run it in the background first and to let an analyst make every decision. *Limitation:* With only 17 fraud cases in the test set, one case changes the result by 5.9 percentage points, so the result is not yet reliable.

**Kyran.** *Kyran.**
    "*Preperation choice:* When I started with the file I first looked at the CSV mannually to see if there were immediate things that stood out, I found the difference in how online was written (ONLINE, Online, online) and used that as a baseline to see if the code to find outliers like this would be correct.
    "*Model finding:* to further optimize the model, if we would have more time, we should look into what variables have the highest impact and use that to train a couple models simultaniously with slight differences to see what models gets the best accuracy.
    "*Business implication:* because it is not known how much it costs the comapny to refund fraudulent transactions and what the cost of running a model like this is I can not conclude if it is worth to use this model over the existing process.

**Alberto Barrionuevo Vargas-Machuca.** *[Write your own paragraph: one preparation choice, one model finding, one business implication.]*